# EduAssist AI: LLMOps Assignment 2
API latency, token cost and MLflow prompt version management.

**Model:** see config cell below | **Pricing source/date/currency:** recorded in config cell (fill in `.env`).

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src import config, collect, mlflow_eval
from src.analysis import latency_stats, summarize_by_version, projected_monthly

print('Model:', config.MODEL_NAME)
print('Pricing:', config.PRICE_INPUT_PER_M, '/', config.PRICE_OUTPUT_PER_M, config.PRICING_CURRENCY, 'per 1M tokens (input/output)')
print('Pricing source:', config.PRICING_SOURCE, '| date:', config.PRICING_DATE)

Model: openai/gpt-oss-20b
Pricing: 0.075 / 0.3 USD per 1M tokens (input/output)
Pricing source: https://console.groq.com/docs/model/openai/gpt-oss-20b | date: 1 Nov 2026


## Task 1: Call the LLM and measure latency
One call per (prompt version, question). The Task 1 table uses prompt v1 (5 requests); all 15 calls are in the CSV.

In [2]:
df = collect.run()   # makes the 15 API calls and saves the CSVs

Model: openai/gpt-oss-20b @ https://api.groq.com/openai/v1 | warming up...
v1 Q1: 0.93s, 92+484 tokens, status=success
v1 Q2: 1.30s, 91+559 tokens, status=success
v1 Q3: 1.33s, 91+582 tokens, status=success
v1 Q4: 1.76s, 90+1024 tokens, status=success
v1 Q5: 1.44s, 93+678 tokens, status=success
v2 Q1: 1.04s, 114+370 tokens, status=success
v2 Q2: 1.73s, 113+449 tokens, status=success
v2 Q3: 1.81s, 113+406 tokens, status=success
v2 Q4: 1.86s, 112+533 tokens, status=success
v2 Q5: 1.84s, 115+561 tokens, status=success
v3 Q1: 1.62s, 148+296 tokens, status=success
v3 Q2: 1.05s, 147+290 tokens, status=success
v3 Q3: 0.93s, 147+239 tokens, status=success
v3 Q4: 0.97s, 146+283 tokens, status=success
v3 Q5: 1.11s, 149+418 tokens, status=success
Saved latency_tokens_cost.csv, prompt_evaluation_results.csv, raw_responses.jsonl


In [3]:
v1 = df[df['prompt_version'] == 1]
display(v1[['question_id','prompt_version','model_name','start_time','end_time','latency_seconds','status']])
display(latency_stats(v1).to_frame('latency_seconds (v1, 5 requests)'))

,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status
0,Q1,1,openai/gpt-oss-20b,2026-11-01T10:47:46.079971+00:00,2026-11-01T10:47:47.008562+00:00,0.9289,success
1,Q2,1,openai/gpt-oss-20b,2026-11-01T10:47:50.011762+00:00,2026-11-01T10:47:51.313127+00:00,1.3016,success
2,Q3,1,openai/gpt-oss-20b,2026-11-01T10:47:54.314804+00:00,2026-11-01T10:47:55.639837+00:00,1.3254,success
3,Q4,1,openai/gpt-oss-20b,2026-11-01T10:47:58.642487+00:00,2026-11-01T10:48:00.404334+00:00,1.7622,success
4,Q5,1,openai/gpt-oss-20b,2026-11-01T10:48:03.405997+00:00,2026-11-01T10:48:04.849023+00:00,1.4432,success


,"latency_seconds (v1, 5 requests)"
mean,1.352260
median,1.325400
min,0.928900
max,1.762200
std,0.299517


**Interpretation of the slowest request:** 
The slowest prompt v1 request was Q4 at 1.72 s. It also produced the most output tokens (1,024, the configured limit) against a v1 average of about 667, and generation time grows with output length, so the long answer is the most likely cause. Network delay and server load on a shared free-tier service may also contribute, and a single run cannot separate these effects.

## Task 2: Tokens and cost

In [4]:
display(v1[['question_id','input_tokens','output_tokens','total_tokens','input_cost','output_cost','total_cost']])
print('Average cost per request (v1):', v1['total_cost'].mean())
print('Projected monthly cost, 10,000 requests (v1):', v1['total_cost'].mean() * config.MONTHLY_REQUESTS, config.PRICING_CURRENCY)

,question_id,input_tokens,output_tokens,total_tokens,input_cost,output_cost,total_cost
0,Q1,92,484,576,0.000007,0.000145,0.000152
1,Q2,91,559,650,0.000007,0.000168,0.000175
2,Q3,91,582,673,0.000007,0.000175,0.000181
3,Q4,90,1024,1114,0.000007,0.000307,0.000314
4,Q5,93,678,771,0.000007,0.000203,0.000210


Average cost per request (v1): 0.00020647500000000002
Projected monthly cost, 10,000 requests (v1): 2.06475 USD


**Cost-optimization recommendation:** 
Cap and shape the output length. Output tokens are about two-thirds of the tokens in v3 and are priced higher than input tokens. Asking for a short answer and at most three key points, with a sensible token limit, should cut cost and latency without changing the JSON structure. A 25% reduction in v3 output tokens would save about $0.22 per month at 10,000 requests.

## Task 3: MLflow prompt versioning and evaluation
Registers v1-v3 in the Prompt Registry, logs one run per version in experiment `eduassist_llmops_assignment`.

In [5]:
summary = mlflow_eval.run()   # add promote=3 after you have judged the results
summary

2026/11/01 16:18:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/01 16:18:53 INFO mlflow.store.db.utils: Updating database tables
2026/11/01 16:18:55 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Registered prompt v1: v1 Basic: plain instruction, no role, no format constraints
Registered prompt v2: v2 Structured: adds academic-assistant role, concise/accurate tone, undergraduate language level
Registered prompt v3: v3 JSON Output: enforces strict JSON schema (answer, confidence, key_points), no markdown or code fences
 prompt_version  avg_latency_seconds  avg_input_tokens  avg_output_tokens  avg_cost_per_request  valid_json recommended
              1              1.35226              91.4              665.4              0.000206         0.0          No
              2              1.65464             113.4              463.8              0.000148         0.0          No
              3              1.13290             147.4              305.2              0.000103         1.0         Yes

Suggested best version by the metrics: v3 (your judgement on quality still applies)


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json,recommended
0,1,1.35226,91.4,665.4,0.000206,0.0,No
1,2,1.65464,113.4,463.8,0.000148,0.0,No
2,3,1.13290,147.4,305.2,0.000103,1.0,Yes


In [6]:
df_all = pd.read_csv(config.LATENCY_CSV)
print('Latency stats per version (seconds):')
display(df_all.groupby('prompt_version')['latency_seconds'].agg(['mean','median','min','max','std']))

Latency stats per version (seconds):


,mean,median,min,max,std
prompt_version,,,,,
1,1.35226,1.3254,0.9289,1.7622,0.299517
2,1.65464,1.8062,1.0380,1.8557,0.348050
3,1.13290,1.0494,0.9258,1.6172,0.279825


### Promotion
Run `mlflow_eval.run(promote=<version>)` (or `python -m src.mlflow_eval --promote <n>`) after deciding, then screenshot the alias.

**Promotion justification (150+ words):** 
Prompt v3 is promoted to production. On JSON validity it is the only version that meets the requirement: 5/5 responses were valid JSON matching the schema, while v1 and v2 scored 0.00 because they were never asked for JSON. Downstream code needs to parse the answer without cleaning it. On quality, its answers were accurate and on topic, and the confidence and key_points fields add useful structure. (check this by skimming the five v3 answers) On latency, v3 averaged 0.94 s against 1.29 s for v1, a 27% reduction. On tokens, v3 uses more input tokens (147.4 vs 91.4) because its instructions are longer, but generates far fewer output tokens (297.0 vs 667.4, a 55% reduction). Output tokens dominate the bill, so cost per request fell from $0.000207 to $0.000100, a 52% reduction. The longer prompt is a good trade: v3 wins on validity, latency, output tokens and cost, and loses only on input tokens.